## Imports

In [ ]:
import os
import pydicom
import SimpleITK as sitk
import numpy as np
import glob
import shutil
from tqdm import tqdm

## Functions

In [ ]:
# This will convert all dicom files in a directory to a single nifti file and copy the pixel spacing and slice thickness attributes

def convert_dicom_to_nifti(dicom_dir, nifti_file):

    # Accept both .IMA and .dcm (case-insensitive)
    dicom_files = [f for f in os.listdir(dicom_dir) if f.lower().endswith(('.ima', '.dcm'))]
    dicom_files.sort()
    if not dicom_files:
        print(f"No DICOM files (.dcm/.IMA) found in {dicom_dir}; skipping.")
        return

    # Read DICOMs
    dicom_slices = [pydicom.dcmread(os.path.join(dicom_dir, f)) for f in dicom_files]
    dicom_data = [s.pixel_array for s in dicom_slices]

    # Convert to SimpleITK image
    nifti_data = sitk.GetImageFromArray(dicom_data)
    nifti_img = sitk.Cast(nifti_data, sitk.sitkUInt16)

    # Try to copy spacing; fall back to defaults if missing
    try:
        y_space, x_space = dicom_slices[0].PixelSpacing
    except Exception:
        y_space = x_space = 1.0
    z_space = getattr(dicom_slices[0], "SliceThickness", 1.0)

    nifti_img.SetSpacing([float(y_space), float(x_space), float(z_space)])

    # Write out nifti
    sitk.WriteImage(nifti_img, nifti_file)


# This will return a list of directories containing DICOM files

def get_directories_with_dicom_files(root_dir):
    dicom_dirs = []
    for dirpath, dirnames, filenames in os.walk(root_dir):
        if any(f.lower().endswith(('.ima', '.dcm')) for f in filenames):
            dicom_dirs.append(dirpath)
    return dicom_dirs

## Generate NIfTI files

In [ ]:
# Given the root directory, this script will find all directories containing DICOM files, convert the DICOM files to a single NIfTI, 
# and save the NIfTI files in the same directory as the DICOM files

root_directory = r"C:\Users\samdg\OneDrive\Documents\Northwestern\Research\ARMS Lab\Jorie\Volume Segmentation Sub-Project\Tenotomy\mz8x5"

dicom_dirs = get_directories_with_dicom_files(root_directory)
for dicom_dir in dicom_dirs:
    dicom_files = [f for f in os.listdir(dicom_dir) if f.endswith('.IMA')]
    dicom_files.sort()
    nifti_file = os.path.join(dicom_dir, os.path.basename(dicom_dir) + ".nii.gz")
    print(nifti_file)
    convert_dicom_to_nifti(dicom_dir, nifti_file)

## Copy NIfTI files to new folder

In [ ]:
# Source and destination directories
source_dir = r'C:\Users\samdg\OneDrive\Documents\Northwestern\Research\ARMS Lab\Jorie\Volume Segmentation Sub-Project\Tenotomy\mz8x5'
destination_dir = r'C:\Users\samdg\OneDrive\Documents\Northwestern\Research\ARMS Lab\Jorie\Volume Segmentation Sub-Project\Tenotomy\mz8x5'

# Create destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Find all NIfTI files in the source directory
nifti_files = glob.glob(os.path.join(source_dir, '**', '*.nii.gz'), recursive=True)

# Copy each NIfTI file to the destination directory
for nifti_file in nifti_files:
    shutil.copy(nifti_file, destination_dir)